# rail3D — the complete pipeline

One notebook for the whole experiment: what the system is, why each design choice was made,
and every command needed to run it end to end.

**How this notebook is built.** Long jobs (dataset generation, the detector sweep) run as
*subprocesses* so the kernel does not accumulate GPU memory and a dropped connection cannot
lose progress — both are resumable, so re-running the cell continues where it stopped.
Everything light (checks, figures, analysis) runs in-kernel so plots appear inline.

**Run order.** Sections 1–3 are checks and context (fast). Section 4 explains the physics.
Sections 5–6 build the dataset. Sections 7–9 train and analyse. Green 'RUN' cells are the
ones that do work; the rest is explanation.

> If you are stepping away during Section 5 (~36 min), prefer the terminal command shown
> there — VS Code disconnects kill notebook cells (though the run is resumable).

## 0. Kernel setup

`RAILDEFECT_DATA_DIR` is read at **import time**, so it must be set before `rail3d` is
imported. If you launched VS Code with `code .` from a shell where you exported it, the
kernel already has it and the line below is a no-op.

In [ ]:
import os
# Set this if the kernel did not inherit it. Windows path with forward slashes.
os.environ.setdefault('RAILDEFECT_DATA_DIR', r'C:/Users/ct2443/Downloads/RailDefect/RailDefect')

import subprocess, sys, json
from pathlib import Path
import numpy as np, torch, matplotlib.pyplot as plt
from IPython.display import Image, display, Markdown

HERE = Path.cwd()
assert (HERE / 'rail3d').is_dir(), f'run this notebook from the rail3D folder (cwd={HERE})'
sys.path.insert(0, str(HERE))

from rail3d import config, data3d, losses3d, mesh3d, optics3d, sections, train3d, viz_setup

PROFILE = 'lab'          # 'laptop' for a small GPU, 'cpu' for no GPU
device = config.get_device(PROFILE)
config.ensure_dirs()

def run(*args, timeout=None):
    """Run a pipeline script as a subprocess, streaming output live.
    Isolation matters: the child frees its GPU memory on exit, so a long
    generation cannot starve a later training cell in this kernel."""
    p = subprocess.Popen([sys.executable, *args], cwd=HERE, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    return p.wait()

def free_gpu():
    """Call between heavy in-kernel phases; the kernel holds tensors alive otherwise."""
    import gc; gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f'GPU reserved: {torch.cuda.memory_reserved(device)/1e9:.2f} GB')

def show(name):
    p = config.FIGURE_DIR / name
    display(Image(filename=str(p))) if p.exists() else print(f'{name} not generated yet')

print(f'device      {device}')
print(f'geometry    grid {config.NX}x{config.NY}, H_MS={config.H_MS} mm, '
      f'segment {config.SEG_LEN} mm, mesh lambda/{config.WVL/config.MESH_DS:.0f}')
print(f'classes     {list(config.CLASS_NAMES)}')

## 1. Pre-flight — is this machine consistent?

**RUN THIS FIRST, EVERY TIME.** Every problem hit during development was a consistency
problem rather than a code bug: stale code on one machine, a checkpoint from when there
were 3 classes, a dataset generated at a different plane height, shards silently reused
across runs because the generator skips files that already exist.

`preflight.py` checks all of it in seconds and prints the exact fix for anything wrong.
Exit code 0 means safe to proceed.

In [ ]:
run('preflight.py', '--profile', PROFILE)

If it reports **STOP**, the most common causes and fixes:

| finding | why it matters | fix |
|---|---|---|
| behind origin | you would run stale code against fresh data | `git checkout -- rail3D/data/ && git pull` |
| `dataset_config.json` missing | the shards predate provenance recording, so their geometry cannot be verified | delete shards, regenerate |
| geometry mismatch | fields were generated for a different plane height / grid / class list — training on them produces numbers that look fine and mean nothing | delete shards, regenerate |
| shards written hours apart | the generator **skips existing shards**, so old and new geometry got mixed | `rm -f data/generated/rail3d_*_shard*.pt`, regenerate in one go |
| checkpoint class-count mismatch | it predates a change to `CLASS_NAMES` and cannot be resumed | `rm -rf data/checkpoints/<run>` |

## 2. What this system is

A **trainable metasurface plus detector array** that identifies rail-head defects from a
single microwave measurement — no digital image is formed. The optics *are* the classifier.

```
horn (224 mm @ 55 deg)
        |  illuminates the rail head
        v
rail surface with a defect       <- physical optics: exact Rayleigh-Sommerfeld surface integral
        |  scatters
        v
measurement plane, 60x30 @ 4 mm, 240 mm above the crown
        |  metasurface: a trainable phase (or pillar-width) map
        v
free-space propagation, 160 mm   <- same exact RS kernel, applied by FFT
        |
        v
detector windows -> 'barcode' -> defect / no-defect + which class
```

The physics is ported from the **experimentally validated** Face3D codebase (a facial-
recognition metasurface at the same 8 mm wavelength), so the solver and propagator are not
new code — they are verified code re-used, with the rail geometry substituted for faces.

**Why a metasurface at all?** Without one, detector windows just integrate whatever the rail
happens to scatter onto them. The metasurface reshapes the field *before* detection so that
defect-relevant differences land where detectors can see them. Section 9 compares against a
no-metasurface baseline to quantify exactly that.

## 3. The measurement geometry, and why

Two choices here were **measured, not inherited**, and both matter more than they look.

### The plane is dark-field on purpose

The horn illuminates at 55 deg, so the specular lobe off a flat crown lands at
`x = -H*tan(55 deg)` — at H=240 mm that is **-343 mm**, far outside the +-120 mm aperture.
The system deliberately collects *off-specular* scatter.

That is not an accident of framing: placing the plane where the lobe **is** captured
(H=80 mm, lobe at -114 mm) collects **13x more energy** and performs **below chance**:

| H (mm) | energy | intact spread | field AUC | det AUC |
|---|---|---|---|---|
| 80 | 1.43e-2 | 0.0203 | **0.567** | **0.419** |
| 160 | 1.06e-3 | 0.0184 | 0.867 | 0.819 |
| **240** | 5.16e-4 | **0.0052** | **0.899** | 0.845 |
| 320 | 3.17e-4 | 0.0083 | 0.890 | 0.842 |
| 480 | 1.60e-4 | 0.0022 | 0.869 | 0.887 |

The specular lobe is dominated by the intact rail's mirror reflection: lots of power, almost
no defect information, and it swamps the signal.

**H=240 was chosen on field AUC** (information physically present at the plane) rather than
det AUC (what a fixed, untrained readout extracts). Training can fix a readout bottleneck;
it cannot recover information that is not there. Beyond 240 the field grows diffuse and
crack speckle washes out — crack field AUC falls 0.803 -> 0.688 — while det AUC rises only
because broader detector footprints are less sensitive to placement jitter. H=240 also keeps
3x more energy than 480, which our *relative* noise model does not penalise but real hardware
would.

Re-derive it any time with `run('scan_geometry.py')` (~1 min).

In [ ]:
run('setup_diagram.py')
show('setup_diagram.png')

Read the middle panel: the metasurface at z=240 mm, detectors at z=400 mm, the horn at
224 mm / 55 deg, and the red x marking where the specular lobe lands — visibly outside the
aperture. The right panel shows the 120 mm rail segment against the 240x120 mm aperture.

**Why the segment is only 120 mm:** a truncation study showed 120 mm preserves every class's
defect signature to >= 0.997 cosine versus a 240 mm reference while being **3.1x faster** to
generate. Truncation shifts the intact field ~5%, but that is common mode — intact and defect
samples share the segment, so it cancels in the comparison. At 80 mm the cut edge falls inside
the illuminated footprint and cracks degrade.

## 4. The defect model

Defects are **per-point depth fields** on the rail surface:

$$\\text{displaced}(s, y) = \\text{intact}(s) - d(s, y)\\,\\hat{n}(s)$$

where `s` runs across the head, `y` along the rail, and `d >= 0` is the depth in mm. An
earlier version blended whole cross-sections per slice, which could only make defects that
were uniform across the head — it could not represent a crack running *along* the rail, or a
compact dent.

| class | footprint | depth | where |
|---|---|---|---|
| `crack` | line divot 10-50 mm long x 2-5 mm wide; longitudinal / transverse / oblique; 30% chance of 2-3 parallel | 2-10 mm | running band + gauge corner |
| `dent` | 2D super-Gaussian 10-30 x 10-30 mm; 20% chance of a pit chain | 1.5-8 mm | running band |
| `wear` | worn cross-section over a 300-900 mm envelope, i.e. the whole segment | 2-8 mm | horn-facing shoulder |
| `shell` | ragged Fourier-modulated ellipse 8-20 mm; 30% chance of a second lobe | 1-5 mm | horn-facing shoulder |

Baselines come from laser-scan measurements (Ye et al. 2018 Table 1; Ye et al. 2023 Fig 9);
the ranges above are the operating ranges for this system, widened from those. Depths are
**sampled**, never clipped — clipping raw CSV depths once pinned 66% of cracks at exactly the
cap, destroying depth diversity.

`y0` (along-track position) is confined to +-10 mm because the sensor rides the train: every
defect passes through the beam centre at some frame. The across-head position `s0` stays
broadly sampled — the train cannot move the sensor sideways.

In [ ]:
viz_setup.mesh_review_figure(save_path=config.FIGURE_DIR / 'mesh_review.png')
plt.show()

## 5. Physics verification

Nine gates, all of which must pass before any result is trustworthy. This is the part that
makes the numbers mean something.

| gate | what it proves |
|---|---|
| V0 | the defect geometry does what it claims (orientations, band confinement, seed reproducibility, resolution independence) |
| V1 | the chunked/batched solver equals the verbatim Face3D integral to ~1e-7 |
| V2 | the FFT propagator equals the verified conv2d kernel to ~1e-6 (~1000x faster) |
| V3 | an independent angular-spectrum propagator agrees to 0.13% |
| V4 | specular lobe lands where optics says, energy is conserved, mesh normals point outward |
| V5 | the 3D solver reproduces the established **2D** pipeline on a uniform rail (r = 0.976) |
| V6 | ray-cast shadowing is real, and free of grazing-ray artifacts |
| V7 | the generation mesh is fine enough (defect-signal cosine 0.9975 vs a 2x finer mesh) |
| V8 | training runs end to end, and a killed run resumes **bit-identically** |

`lab_report.py` runs all of them plus a smoke dataset and writes a paste-able summary.

In [ ]:
run('lab_report.py', '--profile', PROFILE)

## 6. Generate the dataset

~36 min on the 5090 for 4 classes x 5000 + 512 intact, measured at 9.47 samples/s.

**Delete old shards first.** The generator skips shards that already exist — which is what
makes it resumable, and also what silently mixes geometries if stale files are left behind.

Per sample it: builds the swept mesh at lambda/8, ray-casts shadowing against a lambda/2
occluder, evaluates the exact RS surface integral to the measurement plane for the single
bounce (psi1) and the double bounce (psi2), and stores both. The direct horn term (psi0) is
identical for every sample and is cached once.

> If stepping away, run this in a terminal instead:
> `rm -f data/generated/rail3d_*_shard*.pt && python generate_dataset_3d.py --profile lab`

In [ ]:
# Uncomment the delete when you intend to regenerate from scratch.
# for p in (config.GENERATED_DIR).glob('rail3d_*_shard*.pt'): p.unlink()
run('generate_dataset_3d.py', '--profile', PROFILE, '--status')

In [ ]:
run('generate_dataset_3d.py', '--profile', PROFILE)

### Check the data before training on it

In [ ]:
run('inspect_dataset.py', '--root', str(config.GENERATED_DIR))
show('dataset_review.png'); show('dataset_meta.png')

Column 1-2 show each stored sample's geometry **re-derived from its seed**, proving the
shard matches the intended defect. Column 3 is the stored field. Column 4 is the mean
defect-minus-intact intensity — the signature the metasurface has to exploit.

## 7. The objective

The first full run scored a detection AUC of 0.75-0.90 but a **false-alarm rate near 0.5**.
The cause was structural, not a tuning problem: with the required +-4 mm placement
augmentation, the *intact* barcode distribution spreads about as much as the defect signal,
so a fixed absolute margin (the 2D pipeline's approach, valid against a single fixed
reference) is ill-posed. Pass rate and false alarm rose together.

The objective is now **ranking-based**:

| term | purpose |
|---|---|
| `rank` | pairwise hinge over all (defect, intact) pairs — a differentiable AUC surrogate |
| `hardest` | the same hinge on the worst 10% of pairs |
| `intact` | pulls the intact cluster tight |
| `class` | cross-entropy on normalised barcodes |
| `power` | keeps light on the detectors |
| `centroid` | pushes the four class centroids apart |
| `tv` | smoothness on the metasurface map (fabricability) |

Barcodes are per-sample L2-normalised, which removes the global-amplitude component of
placement jitter, and the operating threshold is **calibrated** to a target false-positive
rate on the *validation* intact spread — never on test. Checkpoint selection uses AUC plus
class accuracy, both threshold-independent.

`objective='margin'` restores the old loss exactly, and V8 keeps it running as a regression
guard.

## 8. Train

### How the detectors are optimised

Positions are **trained, not swept**: `SoftDetector2D` holds the window centres as an
`nn.Parameter` behind sigmoid-edged (differentiable) masks, so gradients move them. Variance
pruning runs *inside* the same run. One training run does:

| epochs | what happens |
|---|---|
| 0-40 | full starting grid, soft masks, phase + positions training |
| 40-200 | pruning window: geometric reduction to the final count |
| 200-250 | tau anneal completes, masks sharpen to hard edges |
| 250-1200 | stationary fine-tune at the final count |

All reported metrics use **hard** binary windows, never the soft training masks.

**Why 1200 epochs:** the anneal and pruning make the objective non-stationary for ~250
epochs. A 400-epoch run left only ~150 stationary epochs to settle ~1800 metasurface
parameters. Epochs cost ~0.5 s, so there is no reason to economise. `train()` reports where
the best epoch landed and warns if the model was still improving at the end.

In [ ]:
cfg_slm = train3d.TrainConfig(
    run_name='ms3d_slm_v1', surface='slm', n_epoch=1200,
    batch_size=config.PROFILES[PROFILE].train_batch)
hist_slm = train3d.train(cfg_slm, device=device)

In [ ]:
def plot_history(hist, title):
    fig, ax = plt.subplots(1, 3, figsize=(15, 3.6))
    ax[0].plot([t['loss'] for t in hist['train']]); ax[0].set(title='train loss', xlabel='epoch')
    ax[1].plot([v['auc'] for v in hist['val']], label='AUC')
    ax[1].plot([v['pass_rate'] for v in hist['val']], label='pass@cal')
    ax[1].plot([v['false_alarm'] for v in hist['val']], label='false alarm')
    ax[1].plot([v['class_acc'] for v in hist['val']], label='class acc')
    for p in hist['prune_epochs']: ax[1].axvline(p['epoch'], color='gray', lw=0.5, alpha=0.5)
    ax[1].legend(fontsize=8); ax[1].set(title=f'{title} validation (gray: prunes)', xlabel='epoch')
    ax[2].plot([v['gap_mean'] for v in hist['val']], label='defect gap mean')
    ax[2].plot([v['gap0_mean'] for v in hist['val']], label='intact gap mean')
    ax[2].plot([v['threshold'] for v in hist['val']], 'r--', lw=0.8, label='calibrated thr')
    ax[2].legend(fontsize=8); ax[2].set(title='barcode gaps', xlabel='epoch')
    fig.tight_layout(); plt.show()
plot_history(hist_slm, 'SLM')

### The fabricable version

`MetaUnitSoft` replaces the idealised phase mask with the **real meta-atom** parameterisation:
the trainable variable is a pillar-width map, and amplitude and phase come from per-pixel
polynomial fits to a simulated meta-atom library. Widths use a sigmoid reparameterisation so
gradients stay alive at the [1, 3.8] mm bounds. This is the version you could actually build.

In [ ]:
from dataclasses import replace
cfg_mu = replace(cfg_slm, run_name='ms3d_metaunit_v1', surface='metaunit')
hist_mu = train3d.train(cfg_mu, device=device)
plot_history(hist_mu, 'MetaUnit')

## 9. Results — and where it fails

Aggregate AUC says how well the system does; the analysis below says **which defects it
misses**, by joining each test sample's outcome to the defect parameters stored with it.

**Expect crack to be the limiting class.** It entered training at field AUC 0.803 but det AUC
0.670 — the largest gap between information present and information extracted. Cracks affect
~0.2% of the illuminated surface versus ~8% for wear.

In [ ]:
free_gpu()
run('analyze_results.py', '--run-name', 'ms3d_slm_v1')
show('analysis_parameters.png'); show('analysis_performance.png'); show('analysis_failures.png')

### How few detectors are enough?

Each detector is a physical receiver, so the count is a hardware cost. This starts from a
dense 13x10 = 130-detector grid (~92% plane coverage), lets variance pruning find where the
information actually sits, and trains down to each final count so you can read performance
against receiver count. ~40 min for four counts.

Add `--dist 120 160 200` to sweep the metasurface-to-detector distance at the same time — it
is a training-time propagation, so it needs no regeneration.

In [ ]:
run('sweep_detectors.py', '--counts', '4', '6', '8', '10')
show('detector_sweep.png')

### No-metasurface baseline

The control: identical optics with the metasurface replaced by identity. The difference is
what the metasurface is worth.

In [ ]:
cfg_none = replace(cfg_slm, run_name='ms3d_none_v1', surface='none')
hist_none = train3d.train(cfg_none, device=device)

import pandas as pd
rows = []
for name, surf in [('SLM', 'slm'), ('MetaUnit', 'metaunit'), ('no MS', 'none')]:
    cfg = replace(cfg_slm, run_name={'slm':'ms3d_slm_v1','metaunit':'ms3d_metaunit_v1',
                                     'none':'ms3d_none_v1'}[surf], surface=surf)
    try:
        m, _ = train3d.load_trained(cfg, device, 'best')
        data = train3d.load_all_data(cfg, device)
        ev = train3d.full_evaluation(m, data, cfg)
        rows.append({'model': name, 'AUC': ev['test']['auc'],
                     'class acc': ev['test']['class_acc'],
                     'pass@cal': ev['test']['pass_rate'],
                     'false alarm': ev['test']['false_alarm'],
                     'TPR@1%FPR': ev['roc']['tpr_at_1pct_fpr']})
    except FileNotFoundError:
        print(f'{name}: not trained yet')
pd.DataFrame(rows).set_index('model').round(4) if rows else None

## 10. What to trust, and what to watch

**Trust:** the solver and propagator are verified against the Face3D code they came from
(1e-7, 1e-6) and against the established 2D pipeline (r = 0.976). The dataset records the
geometry it was built with and training refuses to run against a mismatch. A killed training
run resumes bit-identically.

**Watch:**
- **Crack recall** — the hardest class by a wide margin. `analysis_parameters.png` shows
  whether the misses are the shallow, the short, or a particular orientation.
- **Wear being too easy** — it affects ~8% of the surface and separates almost trivially; a
  near-perfect wear column in the confusion matrix is expected, not a sign of overfitting.
- **The noise model is relative** (multiplicative + additive, scaled to signal), so it does
  not penalise absolute energy loss. Configurations that collect less light look better here
  than they would on hardware — this is why H=240 was chosen over 480.
- **Ray-cast shadowing uses a lambda/2 occluder**, which resolves the current defect sizes
  (a real 4.3% effect) but would go inert again if defects were made much finer.

Full detail: `README.md` section 6 ('hard-won findings') and `SETUP_LAB.md`.